In [2]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!pip install -U sentence-transformers
!pip install langchain_community
!pip install peft
!pip install evaluate
!pip install rouge_score
!pip install bert_score
!pip install faiss-gpu

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-9e9hg5e_
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-9e9hg5e_
  Resolved https://github.com/huggingface/transformers to commit bd9dca3b855b5a20ea11097b89c40f34d775f1c7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.0/990.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.5/373.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14

In [3]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

2024-07-22 17:58:50.385671: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-22 17:58:50.385778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-22 17:58:50.513825: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [5]:
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
)

In [6]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00008.bin:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

pytorch_model-00002-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

In [7]:
from bert_score import score
!pip install rouge
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [8]:
import pandas as pd
df = pd.read_csv("/kaggle/input/coovid/covid_abstracts.csv")
df = df["abstract"]
train = pd.read_csv("/kaggle/input/dataset/train.csv")

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=['\n\n']
)

In [10]:
docs = [Document(page_content=entry) for entry in df]
docs = text_splitter.split_documents(docs)


In [11]:
embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-zh-v1.5')

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
faiss_db = FAISS.from_documents(docs, embedding_model)

In [13]:
faiss_retriever = faiss_db.as_retriever(
    search_kwargs={'k': 2}
)

In [14]:
!pip install rank_bm25

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from typing import List

def custom_preprocessing_func(text: str) -> List[str]: # Change the input type to str
    return text.split()

bm25_retriever = BM25Retriever.from_texts(
    [doc.page_content for doc in docs],
    preprocess_func=custom_preprocessing_func
)
bm25_retriever.k = 2

In [16]:
test = pd.read_csv("/kaggle/input/testingdata/testing.csv")

In [17]:
peft_model_id = "ProElectro07/Projectxx2"
model = PeftModel.from_pretrained(base_model, peft_model_id)

adapter_config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

In [18]:
text_generation_pipeline = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

text_generation_pipeline.model = model

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [19]:
tokenizer.pad_token = tokenizer.eos_token

In [20]:
translation_prompt = PromptTemplate(
    template="""
<s>[INST]
You are an experienced linguistic expert and an assitant to a doctor who only understand english.
Your task is to translate the
patient's query to help an english doctor undersand the patient's query.
Here are two examples of good responses:
Example 1:
Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Reponse: Doctor, I have fever and cough, are these things symptoms of covid?
Example2:
Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Translated Query: Can we get covid even after getting vaccinated?
Now, please translate the patient's query below in a similar manner, in English:
Patient's question: ["{input}"]
Provide an accurate translated query which could :
[/INST] </s>
""",
    input_variables=["input"]
)

response_prompt = PromptTemplate(
    template="""
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.
Key points to remember:
1. Provide a direct answer to the patient's question immediately.
2. Use simple, non-technical language that patients can easily understand.
3. Give practical, actionable advice when appropriate.
4. Be reassuring and empathetic, but honest about health concerns.
5. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.
Here are two examples of good responses:
Patient: Doctor, I have fever and cough, are these things symptoms of covid?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.
Patient: Can we get covid even after getting vaccinated?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.
Now, please respond to the patient's query below in a similar manner:
Patient's question: {input}
Provide a relevant, caring, and informative response:
[/INST] </s>
""",
    input_variables=["input"]
)

In [21]:
from langchain.chains import SimpleSequentialChain, LLMChain

In [22]:
class TrimmedOutputChain(SimpleSequentialChain):
    def _call(self, inputs: dict) -> dict:
        _input = inputs[self.input_key]
        for chain in self.chains:
            _input = chain.run(_input)
            # Extract only the generated text, removing any remaining prompt
            _input = _input.split('[/INST]')[-1].strip()
        return {self.output_key: _input}

translation_chain = LLMChain(llm=mistral_llm, prompt=translation_prompt)
response_chain = LLMChain(llm=mistral_llm, prompt=response_prompt)
combined_chain = TrimmedOutputChain(
    chains=[translation_chain, response_chain],
    verbose=True
)

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


In [24]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [25]:
references = []
predictions = []
count = 0
for p, r in zip(test["queries"], test["response"]):
    response = combined_chain({"input": p})
    references.append([r])
    predictions.append(response['output'])  # The output is now in the 'output' key
    count = count + 1
    if count == 10:
        break

# The rest of your evaluation code remains the same
predictions_fixed = predictions
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)
P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)
bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)
print(predictions)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




> Entering new TrimmedOutputChain chain...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



> Finished chain.


> Entering new TrimmedOutputChain chain...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.61 seconds, 16.32 sentences/sec
ROUGE scores: {'rouge1': 0.1753684183161184, 'rouge2': 0.06315363329842524, 'rougeL': 0.13098250282661483, 'rougeLsum': 0.13901269235226904}
BERT scores: {'precision': 0.8316228985786438, 'recall': 0.8951572179794312, 'f1': 0.8621017336845398}
BLEU score: {'bleu': 0.023287478769414047, 'precisions': [0.11579754601226994, 0.03709428129829984, 0.012461059190031152, 0.005494505494505495], 'brevity_penalty': 1.0, 'length_ratio': 5.366255144032922, 'translation_length': 1304, 'reference_length': 243}
["To keep yourself safe from getting coronavirus (COVID-19), here are some steps you can take:\nFirstly, wear a mask in all public settings, especially if social distancing is difficult to maintain. A well-fitted face covering helps prevent respiratory droplets containing virus from traveling through the air and landing on other people's mouths or noses.\nSecondly, practice good hand hygiene by washing your hands often with soap and water for at least 2

In [26]:
translation_prompt = PromptTemplate(
    template="""
<s>[INST]
You are an experienced linguistic expert and an assitant to a doctor who only understand english.
Your task is to translate the
patient's query to help an english doctor undersand the patient's query.
Here are two examples of good responses:
Example 1:
Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Reponse: Doctor, I have fever and cough, are these things symptoms of covid?
Example2:
Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Translated Query: Can we get covid even after getting vaccinated?
Now, please translate the patient's query below in a similar manner, in English:
Patient's question: ["{input}"]
Provide an accurate translated query which could :
[/INST] </s>
""",
    input_variables=["input"]
)

response_prompt = PromptTemplate(
    template="""
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.
Key points to remember:
1. Provide a direct answer to the patient's question immediately.
2. Use simple, non-technical language that patients can easily understand.
3. Give practical, actionable advice when appropriate.
4. Be reassuring and empathetic, but honest about health concerns.
5. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.
Here are two examples of good responses:
Patient: Doctor, I have fever and cough, are these things symptoms of covid?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.
Patient: Can we get covid even after getting vaccinated?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.
Now, please respond to the patient's query below in a similar manner:
Patient's question: {input}

Additional context: {context}

Provide a relevant, caring, and informative response:
[/INST] </s>
""",
    input_variables=["input","context"]
)

In [27]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.6, .4]
)

qa_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": response_prompt}
)

combined_chain = TrimmedOutputChain(
    chains=[translation_chain, response_chain],
    verbose=True
)

references = []
predictions = []
count = 0
for p, r in zip(test["queries"], test["response"]):
    response = combined_chain({"input": p})
    references.append([r])
    predictions.append(response['output'])  # The output is now in the 'output' key

# The rest of your evaluation code remains the same
predictions_fixed = predictions
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)
P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)
bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)
predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




> Entering new TrimmedOutputChain chain...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



> Finished chain.


> Entering new TrimmedOutputChain chain...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 2.20 seconds, 22.69 sentences/sec
ROUGE scores: {'rouge1': 0.1600048430709607, 'rouge2': 0.05650128432469914, 'rougeL': 0.11970163025972042, 'rougeLsum': 0.12560186797917408}
BERT scores: {'precision': 0.8287991285324097, 'recall': 0.8924596905708313, 'f1': 0.8592995405197144}
BLEU score: {'bleu': 0.0225689969507443, 'precisions': [0.09329697143677336, 0.02992626861356079, 0.012669287898645697, 0.0073346046648085665], 'brevity_penalty': 1.0, 'length_ratio': 5.91928632115548, 'translation_length': 6967, 'reference_length': 1177}


["To prevent coronavirus infection, there are several steps you can take. Firstly, practice good hand hygiene by washing your hands frequently with soap and water for at least 20 seconds, especially before eating or touching your face. Secondly, wear a mask in public spaces where physical distancing may be difficult. Thirdly, maintain physical distancing by keeping at least six feet between yourself and others whenever possible. Fourthly, avoid close contact with people who are sick. Finally, cover your mouth and nose with a tissue when coughing or sneezing and dispose of used tissues immediately. It's also essential to stay informed about local guidelines and restrictions on gatherings and travel. Remember, preventing the spread of COVID-19 involves collective effort and responsibility. Stay safe!",
 "Hi there! I'm sorry, could you repeat that last part so I make sure I fully understand what you need help with?",
 "It depends on what is causing the infection. For viral infections such

In [49]:
pip install git+https://github.com/google-research/bleurt.git

  Cloning https://github.com/google-research/bleurt.git to /tmp/pip-req-build-_fmn23kp
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/bleurt.git /tmp/pip-req-build-_fmn23kp
  Resolved https://github.com/google-research/bleurt.git to commit cebe7e6f996b40910cfaa520a63db47807e3bf5c
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.1/352.1 kB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.0 MB/s eta 0:00:00a 0:00:01
  Created wheel for BLEURT: filename=BLEURT-0.0.2-py3-none-any.whl size=16456765 sha256=eaf819fc5db53df659a108177676e8447def8b4607ccfe688eb91a58c0d0dffb
  Stored in directory: /tmp/pip-ephem-wheel-cache-5vy0ih2l/wheels/64/f4/2c/509a6c31b8ebde891a81029fd94f199b1b92f0e7cfc20d417a
Successfully built BLEURT
  Attempting uninstall: keras
    Found existing installation: keras 3.4.1
    Uninstalling keras-3.4.1:
      Successfully uninstalled keras-3.4.1
ERROR:

In [50]:
pip install datasets evaluate

Note: you may need to restart the kernel to use updated packages.


In [51]:
from datasets import load_metric

def calculate_bleurt(predictions, references):
    bleurt = load_metric("bleurt", config_name="BLEURT-20")
    results = bleurt.compute(predictions=predictions, references=references)
    return results['scores']

In [ ]:
bleurt_scores = calculate_bleurt(predictions, references)

# Print individual scores
for pred, ref, score in zip(predictions, references, bleurt_scores):
    print(f"Prediction: {pred}")
    print(f"Reference: {ref}")
    print(f"BLEURT score: {score}\n")

# Calculate and print average score
avg_bleurt = sum(bleurt_scores) / len(bleurt_scores)
print(f"Average BLEURT score: {avg_bleurt}")

/tmp/ipykernel_34/544009198.py:4: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  bleurt = load_metric("bleurt", config_name="BLEURT-20")


The repository for bleurt contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bleurt.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


In [ ]:
import nltk
from nltk.translate import meteor_score

nltk.download('wordnet')
nltk.download('omw-1.4')

def calculate_meteor(prediction, reference):
    return meteor_score.meteor_score([reference], prediction)

# Example usage
meteor_scores = [calculate_meteor(pred, ref) for pred, ref in zip(predictions, references)]
avg_meteor = sum(meteor_scores) / len(meteor_scores)
print(f"Average METEOR score: {avg_meteor}")

In [ ]:
def evaluate_predictions(predictions, references):
    bleurt_scores = calculate_bleurt(predictions, references)
    meteor_scores = [calculate_meteor(pred, ref) for pred, ref in zip(predictions, references)]
    
    avg_bleurt = sum(bleurt_scores) / len(bleurt_scores)
    avg_meteor = sum(meteor_scores) / len(meteor_scores)
    
    return {
        "BLEURT": avg_bleurt,
        "METEOR": avg_meteor,
        "BLEURT_scores": bleurt_scores,
        "METEOR_scores": meteor_scores
    }

In [ ]:
scores = evaluate_predictions(predictions, references)

print(f"Average BLEURT score: {scores['BLEURT']}")
print(f"Average METEOR score: {scores['METEOR']}")
print("Individual BLEURT scores:", scores['BLEURT_scores'])
print("Individual METEOR scores:", scores['METEOR_scores'])